# 📘 RAG NLI Contradiction Checker
This notebook evaluates contradictions between retrieved context documents and generated answers in a Retrieval-Augmented Generation (RAG) pipeline using an NLI model (`facebook/bart-large-mnli`).

In [ ]:
# Install required libraries
!pip install transformers torch pandas

In [1]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
import torch.nn.functional as F
import pandas as pd

C:\Users\aungn\PycharmProjects\ai_rag\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Load pretrained NLI model
model_name = "facebook/bart-large-mnli"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

In [4]:
# Define RAG context and generated answer
query = "Why was the insurance policy cancelled before expiration?"
retrieved_contexts = [
    "The policyholder requested cancellation due to a job relocation.",
    "The claim history triggered an automatic cancellation by the system.",
    "Policy cancellation happened after the insured passed away.",
    "The cancellation was due to non-payment of premiums over 3 months.",
    "There is no record of a cancellation request from the policyholder."
]
generated_answer = "The policy was cancelled because the customer missed several payments."

In [6]:
# Function to get NLI scores
def get_nli_score(premise, hypothesis):
    inputs = tokenizer(premise, hypothesis, return_tensors="pt", truncation=True)
    logits = model(**inputs).logits
    probs = F.softmax(logits, dim=1).detach().numpy().flatten()
    label_map = {0: "contradiction", 1: "neutral", 2: "entailment"}
    return label_map[probs.argmax()], probs

In [7]:
# Run contradiction checks
results = []
for context in retrieved_contexts:
    label, scores = get_nli_score(context, generated_answer)
    results.append({
        "Context": context,
        "Label": label,
        "Contradiction Score": round(scores[0], 4),
        "Neutral Score": round(scores[1], 4),
        "Entailment Score": round(scores[2], 4)
    })
df = pd.DataFrame(results)
df

Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.58.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.


,Context,Label,Contradiction Score,Neutral Score,Entailment Score
0,The policyholder requested cancellation due to...,contradiction,0.9775,0.0222,0.0003
1,The claim history triggered an automatic cance...,neutral,0.0008,0.9986,0.0006
2,Policy cancellation happened after the insured...,contradiction,0.9379,0.0617,0.0004
3,The cancellation was due to non-payment of pre...,entailment,0.0005,0.0330,0.9665
4,There is no record of a cancellation request f...,neutral,0.0120,0.9878,0.0002
